# Example 1: sharpness metrics on a focus scan

**Goal.** Load an experimental image stack, compute frame-wise sharpness metrics, identify the best-focus frame, and inspect its spatial sharpness distribution.

This example shows the two complementary levels of the API:

- `dip.sharpness_stack_stats(...)` for **focus-scan analysis**
- `dip.sharpness_stats(...)` for **single-image analysis**

In [ ]:
__author__ = ['Rafael Celestre']
__contact__ = 'rafael.celestre@esrf.eu'
__license__ = 'CECILL-2.1'
__copyright__ = 'ESRF - The European Synchrotron'
__created__ = '2026.03.06'
__changed__ = '2026.03.09'

import logging
import sys

import barc4dip as dip
import numpy as np

logging.basicConfig(level=logging.INFO, format="%(message)s")

print(sys.executable)
print(sys.version)

%load_ext autoreload
%autoreload 2

# %matplotlib widget
# %matplotlib inline

## 1) Load a focus scan

Replace `data_path` with the location of the example dataset.  
Later this can point to the reproducible dataset archived in Zenodo.

In [ ]:
data_path = "/Users/celestre/work/experimental_data/speckle_tracking/focus_scan/focus_scan_fine.h5"

scan = dip.read_image(data_path, verbose=True)
scan.shape

## 2) Quick visual check

A fast image + histogram view is useful before computing metrics.

In [ ]:
def show_image_summary(image: np.ndarray, *, title: str | None = None,
                       bin_min: int = 0, bin_max: int = 65535, hist: bool = True, ) -> None:
    
    pmin, pmax = 1., 99.

    vmin = np.nanpercentile(image, pmin)
    vmax = np.nanpercentile(image, pmax)

    dip.plotting.plt_image(
        image,
        title=title,
        vmin=vmin,
        vmax=vmax,
        cmap="srw",
        display_origin="lower",
    )

    if hist:
        dip.plotting.plt_histogram(
            image,
            logy=True,
            cumulative=True,
            percentiles=(1, 99),
            bin_min=bin_min,
            bin_max=bin_max,
        )

    dip.plotting.show()

show_image_summary(scan[0], bin_min=100, bin_max=600, title="First frame")


## 3) Compute stack sharpness metrics

The stack function evaluates each frame and, optionally, image tiles.  
Tile statistics are useful to judge whether the apparent focus improvement is spatially uniform across the detector.

In [ ]:
stack_stats = dip.sharpness_stack_stats(scan)

## 4) Compare a few focus indicators

We show one **baseline contrast-like metric** (`stats.std`) and two more classical **sharpness metrics**:

- `gradient.tenengrad`
- `laplacian.var`

For focus selection, gradient- or Laplacian-based metrics are usually the most natural.

In [ ]:
dip.plotting.plt_stack_metric(stack_stats, "stats.std", scope="tiles", uncertainty="band")
dip.plotting.plt_stack_metric(stack_stats, "stats.std", scope="full", uncertainty="none")
dip.plotting.show()

In [ ]:
dip.plotting.plt_stack_metric(stack_stats, "gradient.tenengrad", scope="tiles", uncertainty="band")
dip.plotting.plt_stack_metric(stack_stats, "gradient.tenengrad", scope="full", uncertainty="none")
dip.plotting.show()

In [ ]:
dip.plotting.plt_stack_metric(stack_stats, "laplacian.laplacian_variance", scope="tiles", uncertainty="band")
dip.plotting.plt_stack_metric(stack_stats, "laplacian.laplacian_variance", scope="full", uncertainty="none")
dip.plotting.show()

## 5) Select the best-focus frame

Here we use **Tenengrad** as the focus score.

In [ ]:
best_focus_frame = int(np.argmax(stack_stats["full"]["gradient"]["tenengrad"]))
print(f">>> best focus at frame #{best_focus_frame}")

sharp = scan[best_focus_frame]
vmin = np.nanpercentile(sharp, 1.0)
vmax = np.nanpercentile(sharp, 99.0)

show_image_summary(sharp, title=f"Best-focus frame #{best_focus_frame}", bin_max=1000)


## 6) Compute single-image sharpness statistics

Once the best frame is identified, we can inspect all sharpness blocks in detail.

In [ ]:
focused_stats = dip.sharpness_stats(sharp, verbose=True)
focused_stats

## 7) Spatial distribution across tiles

This plot answers a practical question: **is the image sharp everywhere, or only locally?**

In [ ]:
dip.plotting.plt_tiles_metric(
    sharp,
    focused_stats,
    "gradient.tenengrad",
    show_std=True,
    vmin=vmin,
    vmax=vmax,
    fmt="{:.0f}"
)
dip.plotting.show()

In [ ]:
dip.plotting.plt_tiles_metric(
    sharp,
    focused_stats,
    "gradient.tenengrad",
    show_std=True,
    vmin=vmin,
    vmax=vmax,
    normalize=True)
dip.plotting.show()

## 8) Individual metrics

A few representative metrics can also be called directly.

- **Tenengrad**: gradient-energy sharpness
- **Laplacian variance**: second-derivative sharpness
- **Spectral entropy**: Shannon entropy of the normalized 2D PSD
- **Inverse autocorrelation width**: sharper images tend to have narrower autocorrelation peaks
- **Eigenvalues**: structure-tensor anisotropy / local directional content

In [ ]:
%timeit dip.metrics.sharpness.tenengrad(sharp)
%timeit dip.metrics.sharpness.laplacian_variance(sharp)
%timeit dip.metrics.sharpness.spectral_entropy(sharp)
%timeit dip.metrics.sharpness.inverse_autocorr_width(sharp)
%timeit dip.metrics.sharpness.eigenvalues(sharp)

In [ ]:
tenengrad = dip.metrics.sharpness.tenengrad(sharp, verbose=True)
laplacian = dip.metrics.sharpness.laplacian_variance(sharp, verbose=True)
spectral = dip.metrics.sharpness.spectral_entropy(sharp, verbose=True)
autocorr = dip.metrics.sharpness.inverse_autocorr_width(sharp, verbose=True)
eigenvalues = dip.metrics.sharpness.eigenvalues(sharp, verbose=True)

## 9) Generate a compact report

This is convenient for logbooks, beamtime notes, or regression checks.

In [ ]:
print(dip.logbook_report(focused_stats, 'focused_image.md', complete=True, notes=True))

## Notes

- `stats.std` is useful as a simple contrast-like baseline, but it is not the only nor always the best focus score.
- `gradient.tenengrad` is a good default choice for selecting the sharpest frame.
- Tile statistics are often more informative than full-image metrics alone when illumination or aberrations are not uniform.